# Structured Output
1. Pydantic（推荐）
2. TypeDict
3. JSON Schema
4. @dataclass

## with_structured_output()
- 把结构体转成结构化约束（比如：`JSON Schema`），把返回解析为结构体
- 支持`json_schema`和`function_calling`两种模式。（不同模型默认mode不同）。
- `method=function_calling`原理实际是`tools+tool_choice`强制调用工具,LangChain再解析tool_calls的json为结构体
- `mode='json_schmea'`,大致报文结构（response_format）：
    ```json
      {
          "model": "qwen3.7-plus",
          "messages": [
            {
              "role": "user",
              "content": "请严格解析为JSON: 张三今年30岁已经转成AI架构师了"
            }
          ],
          "response_format": {
            "type": "json_schema",
            "json_schema": {
              "name": "Person",
              "schema": {
                "type": "object",
                "description": "个人信息结构数据",
                "properties": {
                  "name": { "type": "string", "description": "姓名" },
                  "age": { "type": "integer", "description": "年龄", "maximum": 150 },
                  "occu": { "type": "string", "description": "职位" }
                },
                "required": ["name", "age", "occu"]
            }
          }
      }
    }
    ```

In [16]:
from dataclasses import dataclass
from typing import Optional

from common import init_simple_dashscope_model
from pydantic import BaseModel, Field
from rich import print as rprint

# model = init_simple_dashscope_model('qwen3.7-plus')
model = init_simple_dashscope_model('qwen-max')


## Pydantic Structured Output - JSON Schema（默认）
- qwen-max不支持,且在绑定了结构体调用时，提示词必须出现`json`字样，否则报400,
- qwen3.7-plus支持，但需结合提示词`严格解析为JSON`辅助，解析更稳定。如果想保持提示词简洁，结构体的字段名要清晰，不要有歧义。比如`Person`的`user_name`能够很好的解析，但是如果是`name`，模型总是解析成了一个工具名

In [14]:
class Person(BaseModel):
    """个人信息结构数据"""

    # 不用name，qwen3.7-plus会解析为其他的值
    user_name: str = Field(
        description="人名"
    )
    age: int = Field(
        description="年龄",
        le=150
    )
    occu: str = Field(
        description="职位"
    )


model_with_structured = model.with_structured_output(Person)
# resp = model_with_structured.invoke('请严格解析为JSON: 张三今年30岁已经转成AI架构师了')
resp = model_with_structured.invoke('张三今年30岁已经转成AI架构师了')

rprint(resp)

Person(user_name='张三', age=30, occu='AI架构师')

## Pydantic Structured Output - Function Calling
- qwen-max支持
- qwen3.7-plus默认开启了thinking,设置了`method=function_calling`后，本质是通过`tools+tool_choice`强制使用工具，而tool_choice和thinking不能共存。可以尝试关闭深度思考。
- 结构化数据在`convert_to_openai_tool`转为tools的JSON，结构体名称作为工具名，并且通过tool_choice强制调用工具，LLM返回tool_calls,LangChain再通过PydanticToolParser解析为结构体

In [15]:
class Person(BaseModel):
    """个人信息结构数据"""

    # 不用name，会解析为其他的值
    user_name: str = Field(
        description="人名"
    )
    age: int = Field(
        description="年龄",
        le=150
    )
    occu: str = Field(
        description="职位"
    )


model_with_structured = model.with_structured_output(schema=Person, method='function_calling')
# resp = model_with_structured.invoke('请严格解析为JSON: 张三今年30岁已经转成AI架构师了')
resp = model_with_structured.invoke('张三今年30岁已经转成AI架构师了')

rprint(resp)

BadRequestError: Error code: 400 - {'error': {'message': '<400> InternalError.Algo.InvalidParameter: The tool_choice parameter does not support being set to required or object in thinking mode', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_parameter_error'}, 'id': 'chatcmpl-667f5f31-83d7-9ebb-9d6c-baa3867bb03e', 'request_id': '667f5f31-83d7-9ebb-9d6c-baa3867bb03e'}

## Pydantic结构化-高级特性
- 可选字段。当不存在值时，int型最终默认显示0，字符串默认""，设置为`Optinal[Type]`，当值不存在，会显示None。如果之前已经解析出了结果，真实不存在字段值的情况下，模型会产生幻觉。而设置为可选，值缺失会设置为None，更符合实际情况。
- 默认值。 `Field`的`default`设置默认值
- 枚举。`Literal`或者枚举类。value不能是tuple，JSON中没有tuple，模型会解析为list，Pydantic校验就会异常。
- 列表。
- 嵌套
- 限制

### 可选字段、默认值、枚举

In [5]:
from typing import Optional
from enum import Enum
from typing import Literal


class CompanyLocation(Enum):
    BAO_AN = "宝安"
    NAN_SHAN = "南山"


class Person(BaseModel):
    user_name: str = Field(
        description="姓名"
    )
    age: Optional[int] = Field(
        description="年龄"
    )
    gender: str = Field(
        default="男",
        description="性别"
    )
    occu: str = Field(
        description="职位"
    )
    # company_location: CompanyLocation = Field(
    #     default=Live_Lcation.BAO_AN,
    #     description="居住区"
    # )
    company_location: Literal["宝安", "南山"] = Field(
        default="宝安",
        description="公司地址"
    )


model_with_structured = model.with_structured_output(Person, method="function_calling")
resp = model_with_structured.invoke("除了张三，还有人自己在南山创业，成为了董事长")
rprint(resp)

Person(user_name='李四', age=35, gender='男', occu='董事长', company_location='南山')

### 列表、嵌套、限制

In [23]:
from enum import Enum
from pydantic import Field, BaseModel


class Actor(BaseModel):
    actor_name: str = Field(
        description="演员名称"
    )
    country: str = Field(
        description="国籍"
    )


class Movie(BaseModel):
    movie_name: str = Field(
        description="电影名称"
    )
    rating: float = Field(
        default=0,
        # 限制，最高分5.0
        le=5.0,
        description="评分"

    )
    # 列表、嵌套
    actors: list[Actor] = Field(
        description="演员列表"
    )


model_with_structured = model.with_structured_output(Movie, method='function_calling')
# resp = model_with_structured.invoke("醉拳在早年韩国非常火热，平台高达98分，他又中国演员成龙和洪金宝主演")
resp = model_with_structured.invoke("醉拳在早年韩国非常火热，平台高达4.8分，他又中国演员成龙和洪金宝主演")
rprint(resp)

Movie(
    movie_name='醉拳',
    rating=4.8,
    actors=[Actor(actor_name='成龙', country='中国'), Actor(actor_name='洪金宝', country='中国')]
)

## TypedDict - Structured Output
- TypedDict是一个包含类型声明的字典结构，用Annotated附加一些类型意外的额外信息

### 普通TypedDict，校验提示

In [17]:
from typing_extensions import TypedDict


class Movie(TypedDict):
    title: str
    year: int
    director: str
    rating: float


# 没有title字段，编辑器警告
movie_dict: Movie = {
    "title1": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 4.8
}


### TypedDict - Annotated
- `...`表示必填，但最终结果依赖模型

In [26]:
from typing_extensions import TypedDict, Annotated


class Actor(TypedDict):
    actor_name: Annotated[str, ..., "演员名字"]
    role: Annotated[str, ..., "扮演角色"]


class MovieDict(TypedDict):
    title: Annotated[str, ..., "电影名称"]
    year: Annotated[str, ..., "电影的年份"]
    director: Annotated[str, ..., "电影导演"]
    # 嵌套、列表
    cast: Annotated[list[Actor], ..., "演员列表"]
    rating: Annotated[float, ..., "评分,10分制，可包含1位小数"]


resp = model.with_structured_output(MovieDict, method="function_calling").invoke("请介绍一下星际穿越")
rprint(resp)

{
    'title': '星际穿越',
    'year': '2014',
    'director': '克里斯托弗·诺兰',
    'cast': [{'actor_name': '马修·麦康纳', 'role': '库珀'}, {'actor_name': '安妮·海瑟薇', 'role': '布兰德'}],
    'rating': 9.3
}

## @dataclass
- @dataclass会为类生成__init__,__repr__,__eq__等方法，但不同于Python普通类的是，它携带dataclass的元数据信息，会被python标记为数据类，可以充当LangChain的Schema，但是普通类不可以

In [28]:
from dataclasses import dataclass
from pydantic import Field

@dataclass
class Movie():
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

structured_model = model.with_structured_output(Movie, method='function_calling')

response = structured_model.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}
<class 'dict'>


## FakeServer伪造错误结构数据，验证Pydantic的校验
- 验证校验
- 追踪不同模型API报文结构

### DeepSeek验证

In [29]:
from pydantic import BaseModel, Field, SecretStr

from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_base="http://localhost:8889",
    api_key=SecretStr("sk-01323021c664464b8a4187d9444259f7")
)

class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

ValidationError: 2 validation errors for MovieModel
title
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
year
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

### Qwen

In [32]:
from dotenv import load_dotenv
import os
from pathlib import Path
from langchain.chat_models import init_chat_model


# 从.env文件中加载环境变量
load_dotenv(Path('../.env'), override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model='qwen-max',
    model_provider='openai',
    api_key=DASHSCOPE_API_KEY,
    base_url='http://localhost:8889',
)


class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")


model_with_structure = model.with_structured_output(MovieModel, method='function_calling')
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

ValidationError: 2 validation errors for MovieModel
title
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
year
  Field required [type=missing, input_value={'title1': '盗梦空间'...诺兰', 'rating': 9.3}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing